# BM25 on MS MARCO passages

This notebook reimplements the BM25 strategy from `configs/bm25.yml` and runs it on the MS MARCO passage dataset.

Config values:
- k1 = 1.2
- b = 0.75
- title_boost = 9.3
- description_boost = 4.1

MS MARCO passages only contain `description`, so the title field is empty. We still build the title index so the notebook mirrors the config.

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@73ae7d4cd80f
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)    # colab, share data across notebook runs on gdrive
# mount(use_gdrive=False) # <- colab without gdrive
# mount(use_gdrive=False, manual_path="/path/to/directory")  # <- force data path to specific directory, ie you're running locally.


## Load MS MARCO corpus and judgments

The dataset loader will download the MS MARCO passages and small dev qrels on first use.

In [ ]:
import numpy as np
import pandas as pd

from cheat_at_search.msmarco_data import corpus, judgments
from cheat_at_search.tokenizers import snowball_tokenizer

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].to_numpy()

(corpus.shape, judgments.shape)

## Build SearchArray indices

We build separate BM25 indices for title and description and then apply the boosts from the config when scoring.

In [ ]:
from searcharray import SearchArray

title_index = SearchArray.index(
    corpus["title"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)
description_index = SearchArray.index(
    corpus["description"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)

## BM25 strategy implementation

We score each query against title and description using BM25, then combine the scores with the configured boosts.

In [ ]:
from searcharray.similarity import bm25_similarity

def bm25_search(
    query,
    title_index,
    description_index,
    k1=1.2,
    b=0.75,
    title_boost=9.3,
    description_boost=4.1,
    top_k=10,
):
    tokens = snowball_tokenizer(query)
    if not tokens:
        return np.array([], dtype=int), np.array([], dtype=float)

    similarity = bm25_similarity(k1=k1, b=b)
    title_scores = title_index.score(tokens, similarity=similarity)
    description_scores = description_index.score(tokens, similarity=similarity)

    scores = title_boost * title_scores + description_boost * description_scores
    top_k = min(top_k, len(scores))
    if top_k == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
    top_sorted = top_idx[np.argsort(-scores[top_idx])]
    return top_sorted, scores[top_sorted]

## Run BM25 over all queries

Scoring against the full MS MARCO collection is expensive. If you want a faster iteration loop, set `NUM_QUERIES` to a smaller number.

In [ ]:
from cheat_at_search.eval import grade_results, reciprocal_rank

TOP_K = 10
NUM_QUERIES = None  # set to e.g. 200 for faster iteration

queries = judgments[["query_id", "query"]].drop_duplicates().reset_index(drop=True)
if NUM_QUERIES:
    queries = queries.sample(NUM_QUERIES, random_state=42).reset_index(drop=True)

results = []
for row in queries.itertuples(index=False):
    doc_idx, scores = bm25_search(
        row.query,
        title_index,
        description_index,
        top_k=TOP_K,
    )
    for rank, (idx, score) in enumerate(zip(doc_idx, scores), start=1):
        results.append(
            {
                "query_id": row.query_id,
                "query": row.query,
                "doc_id": int(doc_id_lookup[idx]),
                "rank": rank,
                "score": float(score),
            }
        )

results_df = pd.DataFrame(results)
results_df.head()

## Evaluate with NDCG and MRR

In [ ]:
graded = grade_results(judgments, results_df, k=TOP_K)
ndcg = (
    graded.groupby(["query", "query_id"])["discounted_gain"].sum()
    / graded.groupby(["query", "query_id"])["idcg"].first()
).reset_index(name="ndcg")

mrr = reciprocal_rank(graded, max_grade=judgments["grade"].max())

summary = pd.DataFrame(
    {
        "metric": ["NDCG@10", "MRR"],
        "value": [ndcg["ndcg"].mean(), mrr["mrr"].mean()],
    }
)
summary

## Inspect a single query

In [ ]:
sample_query = queries.iloc[0]["query"]
doc_idx, scores = bm25_search(
    sample_query,
    title_index,
    description_index,
    top_k=5,
)

preview = (
    corpus.iloc[doc_idx][["doc_id", "description"]]
    .assign(score=scores)
    .reset_index(drop=True)
)
preview